In [ ]:
%%capture
!pip install timm torch torchvision matplotlib

In [ ]:
!git clone https://github.com/TheGreatRan/diffusion_AMO.git

In [ ]:
%cd diffusion_AMO

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Sampler
from tqdm import tqdm
import os
import time
import shutil
import gc
import random
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.dataset import AmodalDataset
from src.utils.geometry import get_distance_map, compute_occlusion_weight_map
from src.models.pcn_extractor import PCNExtractor
from src.models.denoising_net import DenoisingNetwork, WeightedBCELoss, WeightedIoULoss, BoundaryAwareEdgeLoss
from src.utils.metrics import compute_amodal_metrics

# 3 chiến lược scheduler
from src.schedulers.da_ust import BaselineUSTScheduler, ClampedDAUSTScheduler, ExponentialDAUSTScheduler


def set_seed(seed=42):
    """FIX #7: reproducible training."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
# ============================================================
# ⚙️  CONFIG — ONE-STAGE TRAINING
# ============================================================

# ---- 1. Experiment ----
EXPERIMENT_MODE = "supervised"       # "supervised" | "zero-shot"
STRATEGY_NAME   = "exponential"      # "baseline" | "clamped" | "exponential"
SEED            = 42

# ---- 2. Hyperparameters ----
NUM_EPOCHS     = 40
LEARNING_RATE  = 1e-4
BATCH_SIZE     = 16
IMAGE_SIZE     = (256, 256)
TIMESTEPS      = 1000
MAX_TRAIN_HOURS = 6

OCCLUSION_LOSS_WEIGHT = 15.0
EDGE_LOSS_WEIGHT      = 0
EDGE_LOSS_BOUNDARY_SOURCE = "modal"  # "modal" | "amodal"
EARLY_STOP_PATIENCE   = 0            # 0 = tắt

# ---- 3. Paths ----
COCO_IMG_DIR     = "/kaggle/input/datasets/ralphsitinh/cocoa-image/cocoa_images_extracted"
PIX2GESTALT_DIR  = "/kaggle/input/datasets/ralphsitinh/data-pix2geltat/pix2gestalt_occlusions_release"
COCOA_TRAIN_JSON = "/kaggle/input/datasets/ralphsitinh/coco-amodal-annotations/annotations/COCO_amodal_train2014.json"
COCOA_VAL_JSON   = "/kaggle/input/datasets/ralphsitinh/coco-amodal-annotations/annotations/COCO_amodal_val2014.json"

SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- 4. Resume ----
RESUME_FROM_EXTERNAL = None  # hoặc "/kaggle/input/.../latest_ckpt_*.pth"

# ============================================================
# Derived — không cần sửa
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

if EXPERIMENT_MODE == "zero-shot":
    DATA_DIR_TARGET = PIX2GESTALT_DIR
    MODE_TARGET = "pix2gestalt"
elif EXPERIMENT_MODE == "supervised":
    DATA_DIR_TARGET = COCOA_TRAIN_JSON
    MODE_TARGET = "cocoa_train"
else:
    raise ValueError(f"EXPERIMENT_MODE không hợp lệ: {EXPERIMENT_MODE}")

# FIX #5: BEST_MIOU_CKPT_PATH có mặt ngay từ CONFIG
LATEST_CKPT_PATH   = os.path.join(SAVE_DIR, f"latest_ckpt_{STRATEGY_NAME}_{EXPERIMENT_MODE}.pth")
BEST_CKPT_PATH     = os.path.join(SAVE_DIR, f"best_ckpt_{STRATEGY_NAME}_{EXPERIMENT_MODE}.pth")
BEST_MIOU_CKPT_PATH = os.path.join(SAVE_DIR, f"best_miou_ckpt_{STRATEGY_NAME}_{EXPERIMENT_MODE}.pth")
INDIV_PLOT_PATH    = f"/kaggle/working/loss_chart_{STRATEGY_NAME}_{EXPERIMENT_MODE}.png"

print("🎯 Cấu hình One-stage Training:")
print(f"   Mode                  : {EXPERIMENT_MODE}")
print(f"   Strategy              : {STRATEGY_NAME}")
print(f"   Seed                  : {SEED}")
print(f"   Epochs                : {NUM_EPOCHS}")
print(f"   LR / Batch            : {LEARNING_RATE} / {BATCH_SIZE}")
print(f"   Occlusion weight      : {OCCLUSION_LOSS_WEIGHT}")
print(f"   Edge weight / source  : {EDGE_LOSS_WEIGHT} / {EDGE_LOSS_BOUNDARY_SOURCE}")
print(f"   Early stop patience   : {EARLY_STOP_PATIENCE if EARLY_STOP_PATIENCE > 0 else 'tắt'}")
print(f"   Data dir              : {DATA_DIR_TARGET}")
print(f"   Device                : {DEVICE}")
print(f"   Latest ckpt           : {LATEST_CKPT_PATH}")
print(f"   Best loss ckpt        : {BEST_CKPT_PATH}")
print(f"   Best mIoU_inv ckpt    : {BEST_MIOU_CKPT_PATH}")

In [ ]:
def save_loss_plot_individual(history, save_path, strategy_name, phase_name):
    """FIX #6: dùng os top-level thay vì import cục bộ."""
    plt.figure(figsize=(18, 5))

    # Subplot 1: train loss theo step
    plt.subplot(1, 3, 1)
    plt.plot(history['train_steps'], history['train_losses'], alpha=0.6, color='blue')
    plt.xlabel('Steps'); plt.ylabel('Loss')
    plt.title(f'[{strategy_name.upper()} | {phase_name.upper()}] Train Loss (Steps)')
    plt.grid(True, linestyle='--', alpha=0.5)

    # Subplot 2: train vs val loss theo epoch
    plt.subplot(1, 3, 2)
    if len(history['epoch_train_losses']) > 0:
        epochs = range(1, len(history['epoch_train_losses']) + 1)
        plt.plot(epochs, history['epoch_train_losses'], label='Train', marker='o')
        plt.plot(epochs, history['epoch_val_losses'], label='Val', marker='s')
        plt.xlabel('Epochs'); plt.title('Train vs Val Loss')
        plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)

    # Subplot 3: mIoU_full + mIoU_inv
    plt.subplot(1, 3, 3)
    if len(history.get('epoch_miou_inv', [])) > 0:
        epochs = range(1, len(history['epoch_miou_inv']) + 1)
        plt.plot(epochs, history['epoch_miou_full'], label='mIoU_full', marker='o')
        plt.plot(epochs, history['epoch_miou_inv'],  label='mIoU_inv',  marker='s')
        plt.xlabel('Epochs'); plt.ylabel('mIoU'); plt.ylim(0, 1)
        plt.title('mIoU (fixed t = T-1)')
        plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"📈 Đã lưu biểu đồ tại: {save_path}")

In [ ]:
class ResumableSampler(Sampler):
    """
    Cho phép resume TRONG một epoch (không chỉ ở ranh giới epoch).
    - Mỗi epoch có một permutation cố định (seed + epoch).
    - start_index cho biết vị trí (theo sample) cần bỏ qua khi resume dở dang.
    """
    def __init__(self, data_source, seed=42):
        self.data_source = data_source
        self.seed = seed
        self.epoch = 0
        self.start_index = 0

    def set_epoch(self, epoch, start_index=0):
        self.epoch = epoch
        self.start_index = start_index

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)
        indices = torch.randperm(len(self.data_source), generator=g).tolist()
        return iter(indices[self.start_index:])

    def __len__(self):
        return len(self.data_source) - self.start_index

In [ ]:
def build_system():
    print("\n" + "=" * 50)
    print(f"🚀 KHỞI TẠO HỆ THỐNG: [{STRATEGY_NAME.upper()}] — MODE [{EXPERIMENT_MODE.upper()}]")
    print("=" * 50)

    train_dataset = AmodalDataset(
        data_dir=DATA_DIR_TARGET,
        mode=MODE_TARGET,
        image_size=IMAGE_SIZE,
        coco_img_dir=COCO_IMG_DIR,
    )
    sampler = ResumableSampler(train_dataset, seed=SEED)
    train_dataloader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        sampler=sampler, drop_last=True,
        num_workers=2, pin_memory=True, persistent_workers=True,  # FIX #12
    )

    val_dataset = AmodalDataset(
        data_dir=COCOA_VAL_JSON, mode="cocoa_val",
        image_size=IMAGE_SIZE, coco_img_dir=COCO_IMG_DIR,
    )
    val_dataloader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
        num_workers=2, pin_memory=True,
    )

    if STRATEGY_NAME == "baseline":
        scheduler = BaselineUSTScheduler(num_train_timesteps=TIMESTEPS)
    elif STRATEGY_NAME == "clamped":
        scheduler = ClampedDAUSTScheduler(num_train_timesteps=TIMESTEPS, sigma=5.0, beta_min=0.05)
    elif STRATEGY_NAME == "exponential":
        scheduler = ExponentialDAUSTScheduler(num_train_timesteps=TIMESTEPS, gamma=5.0, beta_min=0.05)
    else:
        raise ValueError(f"Không nhận diện được chiến lược: {STRATEGY_NAME}")

    pcn = PCNExtractor(model_name='pvt_v2_b4', pretrained=True).to(DEVICE)
    dn  = DenoisingNetwork(pcn_channels=pcn.feature_channels, fuse_channels=256).to(DEVICE)

    bce_criterion  = WeightedBCELoss().to(DEVICE)
    iou_criterion  = WeightedIoULoss().to(DEVICE)
    edge_criterion = BoundaryAwareEdgeLoss(weight_boundary=2.0).to(DEVICE)

    optimizer = optim.AdamW([
        {'params': pcn.parameters(), 'lr': LEARNING_RATE * 0.1},
        {'params': dn.parameters(),  'lr': LEARNING_RATE},
    ], weight_decay=1e-2)
    lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    return (train_dataloader, val_dataloader, sampler, scheduler,
            pcn, dn, bce_criterion, iou_criterion, edge_criterion,
            optimizer, lr_scheduler)

In [ ]:
def validate(val_dataloader, scheduler, pcn, dn,
             bce_criterion, iou_criterion, edge_criterion):
    """
    FIX #2: cố định t = TIMESTEPS - 1 (khớp init của eval protocol).

    - Loss: phản ánh worst-case, khớp mức khó khi inference thật.
    - mIoU_full / mIoU_inv: monitor chất lượng one-shot ở mức khó nhất.

    ⚠️  val_loss sẽ TĂNG so với trước (do t khó hơn) — đây là hiện tượng
    bình thường, không phải model tệ đi.
    """
    pcn.eval(); dn.eval()
    val_loss = 0.0
    total_iou_sum, total_iou_count = 0.0, 0
    total_inv_sum, total_inv_count = 0.0, 0

    with torch.no_grad():
        for batch in val_dataloader:
            I, M_v, M_a = [x.to(DEVICE) for x in batch]
            B = I.shape[0]

            # FIX #2: t cố định ở bước khó nhất
            t = torch.full((B,), TIMESTEPS - 1, device=DEVICE).long()

            distance_map = get_distance_map(M_v)
            x_t = scheduler.add_noise(M_a, distance_map, t)
            weight_map = compute_occlusion_weight_map(
                M_a, M_v, occlusion_weight=OCCLUSION_LOSS_WEIGHT
            )

            pyramid_features = pcn(I, M_v, x_t, t, use_hf=False)
            x_hat_0_logits = dn(x_t, t, pyramid_features)

            loss_bce = bce_criterion(x_hat_0_logits, M_a, weight_map)
            loss_iou = iou_criterion(x_hat_0_logits, M_a, weight_map)
            if EDGE_LOSS_WEIGHT > 0:
                boundary_ref = M_a if EDGE_LOSS_BOUNDARY_SOURCE == "amodal" else M_v
                loss_edge = edge_criterion(x_hat_0_logits, M_a, boundary_ref)
            else:
                loss_edge = torch.tensor(0.0, device=DEVICE)
            val_loss += (loss_bce + loss_iou + EDGE_LOSS_WEIGHT * loss_edge).item()

            # mIoU monitoring (one-shot, không phải final inference)
            pred_prob = torch.sigmoid(x_hat_0_logits)
            iou_batch, inv_iou_batch, valid_inv = compute_amodal_metrics(
                pred_prob, M_a, M_v, threshold=0.5
            )
            total_iou_sum += iou_batch.sum().item()
            total_iou_count += iou_batch.numel()
            if valid_inv.any():
                total_inv_sum += inv_iou_batch[valid_inv].sum().item()
                total_inv_count += int(valid_inv.sum().item())

    return (
        val_loss / len(val_dataloader),
        total_iou_sum / max(total_iou_count, 1),
        total_inv_sum / max(total_inv_count, 1),
    )

In [ ]:
def restore_external_checkpoint():
    """Copy checkpoint từ /kaggle/input/... vào SAVE_DIR để resume bình thường."""
    if RESUME_FROM_EXTERNAL is None:
        return

    if not os.path.exists(RESUME_FROM_EXTERNAL):
        print(f"⚠️ CẢNH BÁO: Không tìm thấy RESUME_FROM_EXTERNAL tại: {RESUME_FROM_EXTERNAL}")
        return

    if os.path.exists(LATEST_CKPT_PATH):
        print("⚡ Đã có checkpoint trong Working. Bỏ qua copy từ external.")
        return

    shutil.copy(RESUME_FROM_EXTERNAL, LATEST_CKPT_PATH)
    print(f"✅ Đã copy checkpoint: {LATEST_CKPT_PATH}")

In [ ]:
def train_model(global_start_time, time_limit_seconds):
    (train_dataloader, val_dataloader, sampler, scheduler,
     pcn, dn, bce_criterion, iou_criterion, edge_criterion,
     optimizer, lr_scheduler) = build_system()

    start_epoch = 0
    resume_sample_idx = 0
    best_val_loss  = float('inf')
    best_miou_inv  = 0.0
    epochs_no_improve = 0
    global_step = 0

    history = {
        'train_steps': [], 'train_losses': [],
        'epoch_train_losses': [], 'epoch_val_losses': [],
        'epoch_miou_full': [], 'epoch_miou_inv': [],
    }

    restore_external_checkpoint()

    if os.path.exists(LATEST_CKPT_PATH):
        print(f"♻️ Khôi phục checkpoint [{STRATEGY_NAME} | {EXPERIMENT_MODE}]...")
        checkpoint = torch.load(LATEST_CKPT_PATH, map_location=DEVICE)
        pcn.load_state_dict(checkpoint['pcn_state_dict'])
        dn.load_state_dict(checkpoint['dn_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['lr_scheduler_state_dict'])
        start_epoch       = checkpoint['epoch']
        resume_sample_idx = checkpoint.get('resume_sample_idx', 0)
        best_val_loss     = checkpoint.get('best_val_loss', float('inf'))
        best_miou_inv     = checkpoint.get('best_miou_inv', 0.0)   # FIX #3
        epochs_no_improve = checkpoint.get('epochs_no_improve', 0)
        history           = checkpoint.get('history', history)
        global_step       = checkpoint.get('global_step', 0)

        # FIX #4: đảm bảo history có đủ key (tương thích checkpoint cũ)
        for k in ('epoch_miou_full', 'epoch_miou_inv'):
            history.setdefault(k, [])

        print(f"✅ Khôi phục: Epoch {start_epoch + 1}, "
              f"sample offset {resume_sample_idx}, global_step {global_step}")
        if EARLY_STOP_PATIENCE > 0:
            print(f"   epochs_no_improve = {epochs_no_improve}/{EARLY_STOP_PATIENCE}")
    else:
        print(f"🆕 Không có checkpoint. Bắt đầu [{STRATEGY_NAME} | {EXPERIMENT_MODE}] từ đầu.")

    if start_epoch >= NUM_EPOCHS:
        print(f"🎉 [{STRATEGY_NAME} | {EXPERIMENT_MODE}] đã hoàn thành đủ {NUM_EPOCHS} epochs.")
        return history

    for epoch in range(start_epoch, NUM_EPOCHS):
        if (time.time() - global_start_time) > time_limit_seconds:
            print(f"\n⏰ Hết giờ! Tạm ngắt [{STRATEGY_NAME} | {EXPERIMENT_MODE}]...")
            return history

        offset = resume_sample_idx if epoch == start_epoch else 0
        sampler.set_epoch(epoch, start_index=offset)

        pcn.train(); dn.train()
        train_loss = 0.0

        pbar = tqdm(train_dataloader,
                    desc=f"[{STRATEGY_NAME.upper()}|{EXPERIMENT_MODE.upper()}] Ep {epoch+1}/{NUM_EPOCHS}")
        time_up = False

        for batch_idx, batch in enumerate(pbar):
            I, M_v, M_a = [x.to(DEVICE) for x in batch]
            optimizer.zero_grad()
            B = I.shape[0]

            # Train vẫn random t (giữ distribution đa dạng)
            # t = torch.randint(0, TIMESTEPS, (B,), device=DEVICE).long()
            u = torch.rand(B, device=DEVICE)
            t = (TIMESTEPS * (u ** 0.5)).long().clamp(0, TIMESTEPS - 1)
            distance_map = get_distance_map(M_v)
            weight_map = compute_occlusion_weight_map(
                M_a, M_v, occlusion_weight=OCCLUSION_LOSS_WEIGHT
            )

            x_t = scheduler.add_noise(M_a, distance_map, t)
            pyramid_features = pcn(I, M_v, x_t, t, use_hf=False)
            x_hat_0_logits = dn(x_t, t, pyramid_features)

            loss_bce = bce_criterion(x_hat_0_logits, M_a, weight_map)
            loss_iou = iou_criterion(x_hat_0_logits, M_a, weight_map)
            if EDGE_LOSS_WEIGHT > 0:
                boundary_ref = M_a if EDGE_LOSS_BOUNDARY_SOURCE == "amodal" else M_v
                loss_edge = edge_criterion(x_hat_0_logits, M_a, boundary_ref)
            else:
                loss_edge = torch.tensor(0.0, device=DEVICE)
            loss = loss_bce + loss_iou + EDGE_LOSS_WEIGHT * loss_edge

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(pcn.parameters()) + list(dn.parameters()), 1.0
            )
            optimizer.step()

            train_loss += loss.item()
            global_step += 1

            if global_step % 10 == 0:
                history['train_steps'].append(global_step)
                history['train_losses'].append(loss.item())

            time_up = (time.time() - global_start_time) > time_limit_seconds - 300

            if global_step % 500 == 0 or time_up:
                resume_sample_idx = offset + (batch_idx + 1) * BATCH_SIZE
                torch.save({
                    'mode': EXPERIMENT_MODE,
                    'epoch': epoch,
                    'resume_sample_idx': resume_sample_idx,
                    'global_step': global_step,
                    'pcn_state_dict': pcn.state_dict(),
                    'dn_state_dict': dn.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'lr_scheduler_state_dict': lr_scheduler.state_dict(),
                    'best_val_loss': best_val_loss,
                    'best_miou_inv': best_miou_inv,      # FIX #3: có mặt trong mid-epoch
                    'epochs_no_improve': epochs_no_improve,
                    'history': history,
                }, LATEST_CKPT_PATH)

            pbar.set_postfix({"Loss": f"{loss.item():.4f}"})

            if time_up:
                print(f"\n⏰ Hết giờ giữa epoch! Đã lưu checkpoint tại sample {resume_sample_idx}.")
                return history

        # ----- Hết epoch -----
        avg_train_loss = train_loss / len(train_dataloader)
        avg_val_loss, miou_full, miou_inv = validate(
            val_dataloader, scheduler, pcn, dn,
            bce_criterion, iou_criterion, edge_criterion
        )
        lr_scheduler.step()

        history['epoch_train_losses'].append(avg_train_loss)
        history['epoch_val_losses'].append(avg_val_loss)
        history['epoch_miou_full'].append(miou_full)
        history['epoch_miou_inv'].append(miou_inv)

        # FIX #8: in log đầy đủ
        print(f"📊 [{STRATEGY_NAME.upper()}|{EXPERIMENT_MODE.upper()}] Epoch {epoch+1}: "
              f"Train = {avg_train_loss:.5f} | Val = {avg_val_loss:.5f} "
              f"| mIoU_full = {miou_full:.4f} | mIoU_inv = {miou_inv:.4f}")

        save_loss_plot_individual(history, INDIV_PLOT_PATH, STRATEGY_NAME, EXPERIMENT_MODE)

        # ----- Update best -----
        improved_loss = avg_val_loss < best_val_loss
        if improved_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        improved_miou = miou_inv > best_miou_inv
        if improved_miou:
            best_miou_inv = miou_inv

        # ----- Save checkpoint (LATEST + BEST_LOSS + BEST_MIOU) -----
        checkpoint_data = {
            'mode': EXPERIMENT_MODE,
            'epoch': epoch + 1,
            'resume_sample_idx': 0,
            'global_step': global_step,
            'pcn_state_dict': pcn.state_dict(),
            'dn_state_dict': dn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'lr_scheduler_state_dict': lr_scheduler.state_dict(),
            'best_val_loss': best_val_loss,
            'best_miou_inv': best_miou_inv,
            'epochs_no_improve': epochs_no_improve,
            'history': history,
        }
        torch.save(checkpoint_data, LATEST_CKPT_PATH)

        if improved_loss:
            torch.save(checkpoint_data, BEST_CKPT_PATH)
            print(f"🏆 Best loss checkpoint: {best_val_loss:.5f}")

        if improved_miou:
            torch.save(checkpoint_data, BEST_MIOU_CKPT_PATH)
            print(f"🏆 Best mIoU_inv checkpoint: {best_miou_inv:.4f}")

        # ----- Early stopping -----
        if EARLY_STOP_PATIENCE > 0 and epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"\n🛑 EARLY STOPPING tại epoch {epoch+1} "
                  f"(best_val_loss={best_val_loss:.5f}, best_miou_inv={best_miou_inv:.4f}).")
            del pcn, dn, optimizer, scheduler, train_dataloader, val_dataloader
            torch.cuda.empty_cache(); gc.collect()
            return history

    print(f"\n🎉 [{STRATEGY_NAME} | {EXPERIMENT_MODE}] đã hoàn thành đủ {NUM_EPOCHS} epochs!")
    del pcn, dn, optimizer, scheduler, train_dataloader, val_dataloader
    torch.cuda.empty_cache(); gc.collect()
    return history

In [ ]:
def main():
    global_start_time = time.time()
    time_limit_seconds = MAX_TRAIN_HOURS * 3600

    history = train_model(global_start_time, time_limit_seconds)

    print("\n🏁 KẾT THÚC LẦN CHẠY NÀY.")
    print(f"👉 Checkpoint mới nhất   : {LATEST_CKPT_PATH}")
    print(f"👉 Best loss checkpoint   : {BEST_CKPT_PATH}")
    print(f"👉 Best mIoU checkpoint   : {BEST_MIOU_CKPT_PATH}")
    return history


history = main()